In [ ]:
import sys

sys.path.append("../..")
from setup_figs import pd, sns, np, plt, stats, clean_names

In [ ]:
fi = pd.read_parquet("0.parquet")
fi = clean_names(fi[fi.split == "test"])
spearman = pd.read_parquet("1.parquet")
spearman = clean_names(spearman[spearman.split == "test"])
weights = clean_names(pd.read_parquet("2.parquet"))
weights["AbsWeight"] = weights.Weight.abs()
weights["AbsGTWeight"] = weights.GTWeight.abs()

In [ ]:
id_cols = [
    "trainer.model_builder.param",
    "trainer.dataset.noise_level",
    "trainer.dataset.seed",
]
hue_order = ["MLEM", "FR-RSA-I"]

# Frobenius

## Main

In [ ]:
ax = sns.lineplot(
    weights[weights.Feature == "Feat. 1"],
    x="trainer.dataset.noise_level",
    y="L2",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.get_legend().set_title(None)
ax.set_ylabel("Frobenius distance")
_ = ax.set_xlabel("Noise level")
sns.despine(trim=True)
plt.title("Weights distance to ground-truth", pad=10)
plt.savefig("../../paper/figs/simulation/fro_gt.pdf", bbox_inches="tight")
plt.show()

# Number of steps

In [ ]:
ax = sns.lineplot(
    weights[weights.Feature == "Feat. 1"],
    x="trainer.dataset.noise_level",
    y="n_epochs",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.get_legend().set_title(None)
ax.set_ylabel("# training steps")
_ = ax.set_xlabel("Noise level")
sns.despine(trim=True)
plt.savefig("../../paper/figs/simulation/n_epochs.pdf", bbox_inches="tight")
plt.show()

## Diagonal vs off diagonal

In [ ]:
weights["Type"] = np.where(weights.Feature.str.contains("x"), "Interaction", "Feature")

In [ ]:
L2_per_type = (
    weights.groupby(id_cols + ["Type"])
    .apply(
        lambda x: (x.GTWeight - x.Weight).abs().mean(),
        include_groups=False,
    )
    .reset_index(name="L2")
)

In [ ]:
# ax = sns.lineplot(
#     L2_per_type.rename(columns={"trainer.model_builder.param": "Param"}),
#     x="trainer.dataset.noise_level",
#     y="L2",
#     hue="Param",
#     style="Type",
#     marker="o",
#     errorbar="sd",
# )
# ax.get_legend().set_title(None)
# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
# ax.set_ylabel("Avg dist. to g.t.")
# ax.set_xlabel("Noise level")

# Spearman

In [ ]:
ax = sns.lineplot(
    spearman,
    x="trainer.dataset.noise_level",
    y="mean",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.get_legend().set_title(None)
ax.set_xlabel("Noise level")
ax.set_ylabel(r"Test Spearman $\rho$")
sns.despine(trim=True)
plt.savefig("../../paper/figs/simulation/spearman.pdf", bbox_inches="tight")
plt.show()

# Weighted $\tau$

##  w.r.t. ground-truth

In [ ]:
# weightedtau = (
#     weights.groupby(
#         id_cols,
#     )
#     .apply(
#         lambda x: stats.weightedtau(x.GTWeight.abs(), x.Weight.abs()).statistic,
#         include_groups=False,
#     )
#     .reset_index(name="Weighted $\\tau$")
# )

In [ ]:
# ax = sns.lineplot(
#     weightedtau.rename(columns={"trainer.model_builder.param": "Param"}),
#     x="trainer.dataset.noise_level",
#     y="Weighted $\\tau$",
#     hue="Param",
#     hue_order=hue_order,
#     style="Param",
#     dashes=False,
#     markers=True,
#     errorbar="sd",
# )
# ax.get_legend().set_title(None)
# sns.despine(trim=True)
# ax.set_xlabel("Noise level")
# ax.set_title("Weighted $\\tau$ between\nabsolute weights and ground-truth", pad=15)
# plt.savefig("../../paper/figs/simulation/weighted_tau_gt.pdf")
# plt.show()

##  FI/weights

In [ ]:
# weightedtau = weights.merge(fi, on=id_cols + ["Feature"])
# weightedtau = (
#     weightedtau.groupby(
#         id_cols,
#     )
#     .apply(
#         lambda x: stats.weightedtau(x.Weight.abs(), x["mean"].abs()).statistic,
#         include_groups=False,
#     )
#     .reset_index(name="Weighted $\\tau$")
# )

In [ ]:
# ax = sns.lineplot(
#     weightedtau,
#     x="trainer.dataset.noise_level",
#     y="Weighted $\\tau$",
#     hue="trainer.model_builder.param",
#     hue_order=hue_order,
#     style="trainer.model_builder.param",
#     markers=True,
#     dashes=False,
#     errorbar="sd",
# )
# sns.move_legend(ax, "upper right", bbox_to_anchor=(1, 1.1), title=None)
# ax.set_xlabel("Noise level")
# ax.set_title("Weighted $\\tau$ between\nabsolute weights and FIs", pad=35)
# sns.despine(trim=True)
# plt.savefig(
#     "../../paper/figs/simulation/weighted_tau_weights_fis.pdf", bbox_inches="tight"
# )
# plt.show()

## MLEM/FR-RSA

In [ ]:
# weights_dist = weights[weights["trainer.model_builder.param"] == hue_order[0]]
# weights_dist = weights_dist.merge(
#     weights[weights["trainer.model_builder.param"] == hue_order[1]],
#     on=["Feature", "trainer.dataset.noise_level", "trainer.dataset.seed"],
# )
# weights_dist = (
#     weights_dist.groupby(
#         ["trainer.dataset.noise_level", "trainer.dataset.seed"],
#     )
#     .apply(
#         lambda x: ((x.Weight_x - x.AbsWeight_y) ** 2).sum() ** 0.5,
#         include_groups=False,
#     )
#     .reset_index(name="Distance")
# )
# weights_dist["Type"] = "Weights"
# distances = [weights_dist]
# fis_dist = fi[fi["trainer.model_builder.param"] == hue_order[0]]
# fis_dist = fis_dist.merge(
#     fi[fi["trainer.model_builder.param"] == hue_order[1]],
#     on=["Feature", "trainer.dataset.noise_level", "trainer.dataset.seed"],
# )
# fis_dist = (
#     fis_dist.groupby(
#         ["trainer.dataset.noise_level", "trainer.dataset.seed"],
#     )
#     .apply(
#         lambda x: stats.weightedtau(x["mean_x"], x["mean_y"]).statistic,
#         include_groups=False,
#     )
#     .reset_index(name="Distance")
# )
# fis_dist["Type"] = "FIs"
# distances.append(fis_dist)
# distances = pd.concat(distances)

In [ ]:
# ax = sns.lineplot(
#     distances,
#     x="trainer.dataset.noise_level",
#     y="Distance",
#     hue="Type",
#     style="Type",
#     markers=True,
#     dashes=False,
#     errorbar="sd",
# )
# ax.get_legend().set_title(None)
# ax.set_xlabel("Noise level")
# ax.set_title("Distance between\nMLEM and FR-RSA-I", pad=10)
# sns.despine(trim=True)
# plt.savefig("../../paper/figs/simulation/mlem_frrsa.pdf", bbox_inches="tight")
# plt.show()